<a href="https://colab.research.google.com/github/AndresAvella09/Proyecto_Multimodal/blob/main/notebooks/07_tmdb_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 — Pósters y sinopsis desde TMDB → embeddings

El notebook 05 dejó dos **listas de películas** (sin pósters ni sinopsis):

| Grupo | Archivo del 05 | Qué son | Para qué |
|---|---|---|---|
| `control` | `control_viejas.parquet` | ~1500 películas viejas del **test** de MM-IMDB | separar el efecto de la **fuente** |
| `reciente` | `recientes_seleccion.parquet` | ~3000 películas **2020–2026** | medir el efecto de la **época** |

Este notebook baja de **TMDB** el póster y la sinopsis de cada una, los procesa **igual que MM-IMDB**
y los pasa por **los mismos encoders del 03** (código compartido en `src/embeddings.py`). No entrena
ni evalúa: deja los embeddings listos para el notebook 08, que aplica los modelos guardados en el 06.

**Procesamiento idéntico a MM-IMDB:**
- Póster: redimensionado a 256×160 (la forma del HDF5) → relleno a cuadrado con la media de CLIP → CLIP.
- Sinopsis: minúsculas, números como `N`, guiones separados (estilo de MM-IMDB tras el notebook 01)
  → CLIP texto por fragmentos y DistilBERT.

Lo que no se puede igualar (estilo de las sinopsis de TMDB, calidad y versión del póster,
interpolación usada originalmente en MM-IMDB) es justamente el **efecto de la fuente** que mide el
grupo control.

**Requisitos:**
1. Una llave de la API de TMDB (gratuita, uso no comercial): <https://www.themoviedb.org/settings/api>.
   Sirve tanto la *API Key* (v3) como el *API Read Access Token* (v4).
2. Guardarla en Colab → 🔑 **Secrets** con el nombre `TMDB_API_KEY` y activar el acceso del notebook.
   **No la escriban en el código**: el notebook queda en GitHub.
3. GPU recomendada para los embeddings (en CPU tarda ~15 min). La descarga tarda ~5–10 min la primera
   vez; todo queda cacheado en Drive y se puede retomar si Colab se corta.

*This product uses the TMDB API but is not endorsed or certified by TMDB.*

In [ ]:
import os, sys, json, time, subprocess
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

REPO = '/content/Proyecto_Multimodal'
if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/AndresAvella09/Proyecto_Multimodal.git', REPO], check=True)
sys.path.insert(0, REPO)

import numpy as np, pandas as pd, torch, requests
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image
from src.embeddings import (Encoders, redimensionar_como_mmimdb, rellenar_cuadrado,
                            normalizar_sinopsis)

DATA = '/content/drive/MyDrive/proyecto_multimodal/data'
EMB = '/content/drive/MyDrive/proyecto_multimodal/embeddings'
TMDB_DIR = os.path.join(DATA, 'tmdb')
POSTERS = os.path.join(TMDB_DIR, 'posters')
os.makedirs(POSTERS, exist_ok=True)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3, 'figure.dpi': 110})
COLORES = {'mmimdb': '#2563eb', 'control': '#10b981', 'reciente': '#ea580c'}

## 1. Lista de películas a bajar

Se unen los dos grupos en una sola tabla con las **etiquetas de IMDb hoy** (`hoy_<género>`), que es
la fuente de etiquetas común de la comparación temporal (decisión del notebook 05).

In [ ]:
con_anio = pd.read_parquet(f'{DATA}/mmimdb_con_anio.parquet')
GENEROS = list(con_anio.columns[4:27])
HOY = [f'hoy_{g}' for g in GENEROS]
with open(f'{DATA}/generos_comparacion.json') as fh:
    GENEROS_COMP = json.load(fh)['generos_comparacion']

rec = pd.read_parquet(f'{DATA}/recientes_seleccion.parquet')
rec = rec.rename(columns=dict(zip(GENEROS, HOY)))   # en las recientes, GENEROS ya son las de IMDb hoy
rec['grupo'], rec['fila_hdf5'] = 'reciente', -1

ctrl = pd.read_parquet(f'{DATA}/control_viejas.parquet')
ctrl = ctrl.merge(con_anio[['fila_hdf5', 'numVotes', 'averageRating']], on='fila_hdf5', how='left')
ctrl['grupo'] = 'control'

COLS = ['grupo', 'tconst', 'titulo', 'anio', 'numVotes', 'averageRating', 'fila_hdf5'] + HOY
obj = pd.concat([ctrl[COLS], rec[COLS]], ignore_index=True)
assert obj['tconst'].is_unique, "una película no puede estar en los dos grupos"
print(obj.groupby('grupo').agg(peliculas=('tconst', 'size'), anio_mediano=('anio', 'median'),
                               votos_medianos=('numVotes', 'median')))

## 2. Consultar TMDB

Endpoint `/find/{imdb_id}?external_source=imdb_id`: dado el `tconst` de IMDb devuelve la película
de TMDB con su sinopsis (`overview`, en inglés) y la ruta del póster. Cada respuesta se guarda en
`tmdb/respuestas.jsonl` apenas llega: si Colab se corta, al volver a correr solo se consultan las
que faltan. Se usan 8 consultas en paralelo y se respeta el límite de TMDB (respuesta 429 → esperar).

In [ ]:
def leer_llave():
    llave = os.environ.get('TMDB_API_KEY')
    if not llave and EN_COLAB:
        from google.colab import userdata
        try:
            llave = userdata.get('TMDB_API_KEY')
        except Exception as e:
            raise RuntimeError("Falta el secreto TMDB_API_KEY en Colab (🔑 Secrets) o no tiene acceso "
                               "habilitado para este notebook.") from e
    if not llave:
        raise RuntimeError("Falta la llave de TMDB: definir el secreto TMDB_API_KEY.")
    return llave.strip()

LLAVE = leer_llave()
API = 'https://api.themoviedb.org/3'
sesion = requests.Session()
if LLAVE.startswith('eyJ'):      # token v4 (JWT) -> cabecera Bearer
    sesion.headers['Authorization'] = f'Bearer {LLAVE}'
    PARAMS_AUTH = {}
else:                            # API key v3 -> parámetro
    PARAMS_AUTH = {'api_key': LLAVE}
del LLAVE

def pedir(url, params=None, intentos=5):
    for k in range(intentos):
        try:
            r = sesion.get(url, params=params, timeout=20)
        except requests.RequestException:
            time.sleep(2 ** k); continue
        if r.status_code == 429:
            time.sleep(float(r.headers.get('Retry-After', 2 ** k))); continue
        if r.status_code >= 500:
            time.sleep(2 ** k); continue
        return r
    raise RuntimeError(f"TMDB no respondió tras {intentos} intentos: {url}")

def consultar(tconst):
    r = pedir(f'{API}/find/{tconst}', {'external_source': 'imdb_id', 'language': 'en-US', **PARAMS_AUTH})
    if r.status_code == 401:
        raise RuntimeError("TMDB rechazó la llave (401). Revisar el secreto TMDB_API_KEY.")
    if r.status_code != 200:
        return {'tconst': tconst, 'estado': f'http_{r.status_code}'}
    pelis = r.json().get('movie_results', [])
    if not pelis:
        return {'tconst': tconst, 'estado': 'no_encontrada'}
    p = pelis[0]
    return {'tconst': tconst, 'estado': 'ok', 'tmdb_id': p.get('id'), 'titulo_tmdb': p.get('title'),
            'fecha_tmdb': p.get('release_date'), 'overview': p.get('overview') or '',
            'poster_path': p.get('poster_path')}

ruta_resp = os.path.join(TMDB_DIR, 'respuestas.jsonl')
hechas = {}
if os.path.exists(ruta_resp):
    with open(ruta_resp, encoding='utf-8') as fh:
        for linea in fh:
            d = json.loads(linea); hechas[d['tconst']] = d
faltan = [t for t in obj['tconst'] if t not in hechas]
print(f"Ya consultadas: {len(hechas)} | faltan: {len(faltan)}")

t0 = time.time()
with open(ruta_resp, 'a', encoding='utf-8') as fh, ThreadPoolExecutor(max_workers=8) as ex:
    futuros = [ex.submit(consultar, t) for t in faltan]
    for k, fut in enumerate(as_completed(futuros), 1):
        d = fut.result(); hechas[d['tconst']] = d
        fh.write(json.dumps(d, ensure_ascii=False) + '\n')
        if k % 500 == 0:
            fh.flush(); print(f"  {k}/{len(faltan)}  ({time.time() - t0:.0f} s)")
print(f"Consultas listas en {time.time() - t0:.0f} s")

resp = pd.DataFrame([hechas[t] for t in obj['tconst']])
tab = obj[['grupo', 'tconst']].merge(resp, on='tconst')
print(tab.groupby('grupo')['estado'].value_counts().unstack(fill_value=0))

## 3. Descargar los pósters

Se baja la versión de 342 px de ancho (`w342`): alcanza de sobra para 256×160 y pesa poco. Quedan
en `tmdb/posters/<tconst>.jpg`.

In [ ]:
IMG = 'https://image.tmdb.org/t/p/w342'

def bajar_poster(tconst, ruta_tmdb):
    destino = os.path.join(POSTERS, f'{tconst}.jpg')
    if os.path.exists(destino):
        return tconst, True
    r = pedir(IMG + ruta_tmdb)
    if r.status_code != 200 or not r.content:
        return tconst, False
    with open(destino, 'wb') as fh:
        fh.write(r.content)
    return tconst, True

con_poster = tab[(tab['estado'] == 'ok') & tab['poster_path'].notna()]
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    res = dict(f.result() for f in as_completed(
        [ex.submit(bajar_poster, t, p) for t, p in zip(con_poster['tconst'], con_poster['poster_path'])]))
tab['poster_ok'] = tab['tconst'].map(res).fillna(False).astype(bool)
print(f"Pósters en {time.time() - t0:.0f} s")
print(tab.groupby('grupo')['poster_ok'].mean().rename('% con póster').map('{:.1%}'.format))

## 4. Qué películas quedan y si las que se pierden son distintas

Una película entra si TMDB la encontró, tiene **sinopsis en inglés no vacía** y **póster**. Si las
que se pierden son sistemáticamente distintas (más viejas, menos votadas), el grupo deja de
representar lo que el 05 eligió: se compara año y votos de incluidas vs excluidas.

In [ ]:
tab = obj.merge(tab.drop(columns=['grupo']), on='tconst')
tab['sinopsis_ok'] = tab['overview'].fillna('').str.strip().str.len() > 0
tab['incluida'] = (tab['estado'] == 'ok') & tab['sinopsis_ok'] & tab['poster_ok']

# Chequeo de identidad: el año de TMDB debería coincidir con el de IMDb (±1 por estrenos escalonados)
anio_tmdb = pd.to_numeric(tab['fecha_tmdb'].fillna('').str[:4], errors='coerce')
tab['anio_dif'] = (anio_tmdb - tab['anio']).abs()

resumen = tab.groupby('grupo').agg(
    total=('tconst', 'size'), encontradas=('estado', lambda x: (x == 'ok').mean()),
    con_sinopsis=('sinopsis_ok', 'mean'), con_poster=('poster_ok', 'mean'), incluidas=('incluida', 'mean'),
    anio_difiere_mas_de_1=('anio_dif', lambda x: (x > 1).mean()))
print(resumen.to_string(float_format='{:.1%}'.format))

print("\nIncluidas vs excluidas (medianas):")
print(tab.groupby(['grupo', 'incluida'])[['anio', 'numVotes']].median().to_string())

## 5. Normalizar las sinopsis y mirar los datos

Para el grupo control existe la **misma película en las dos fuentes**: se muestran lado a lado el
póster y la sinopsis de MM-IMDB y de TMDB. Es la forma más directa de ver qué es el "efecto de la
fuente".

In [ ]:
dat = tab[tab['incluida']].reset_index(drop=True)
dat['sinopsis'] = dat['overview'].map(normalizar_sinopsis)

meta_mm = pd.read_parquet(f'{DATA}/mmimdb_meta.parquet')
sinopsis_mm = meta_mm.set_index('fila_hdf5')['sinopsis']

def leer_poster(tconst):
    return redimensionar_como_mmimdb(Image.open(os.path.join(POSTERS, f'{tconst}.jpg')))

ejemplos = dat[dat['grupo'] == 'control'].sample(3, random_state=SEED)
for _, r in ejemplos.iterrows():
    print(f"=== {r['titulo']} ({int(r['anio'])}) ===")
    print("MM-IMDB:", sinopsis_mm[r['fila_hdf5']][:300])
    print("TMDB   :", r['sinopsis'][:300], "\n")

fig, axes = plt.subplots(1, 6, figsize=(13, 4))
for ax, (_, r) in zip(axes, dat.groupby('grupo').sample(3, random_state=SEED).iterrows()):
    ax.imshow(rellenar_cuadrado(leer_poster(r['tconst']))); ax.axis('off')
    ax.set_title(f"{r['grupo']}\n{str(r['titulo'])[:18]} ({int(r['anio'])})", fontsize=8)
plt.tight_layout(); plt.show()

## 6. Embeddings con los mismos encoders del 03

**Chequeo previo de que `src/embeddings.py` reproduce el notebook 03:** para las películas control se
recalculan los embeddings de texto **de su sinopsis de MM-IMDB** y se comparan con los guardados por el
03. La similitud coseno debe ser ≈ 1 (pequeñas diferencias por GPU/CPU). Si no lo es, el código
compartido no es el mismo procedimiento y cualquier comparación posterior quedaría contaminada.

In [ ]:
enc = Encoders(device)

filas = dat.loc[dat['grupo'] == 'control', 'fila_hdf5'].values[:200]
textos_mm = sinopsis_mm.loc[filas].tolist()
def cos(a, b): return (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
c_clip = cos(enc.texto_clip(textos_mm), np.load(f'{EMB}/clip_txt.npy', mmap_mode='r')[filas])
c_bert = cos(enc.texto_bert(textos_mm), np.load(f'{EMB}/distilbert_txt.npy', mmap_mode='r')[filas])
print(f"Reproducción del 03 | CLIP texto: cos mín {c_clip.min():.5f} | DistilBERT: cos mín {c_bert.min():.5f}")
assert c_clip.min() > 0.999 and c_bert.min() > 0.999, "src/embeddings.py no reproduce el notebook 03"

In [ ]:
rutas = {m: f'{EMB}/tmdb_{m}.npy' for m in ['clip_img', 'clip_txt', 'distilbert_txt']}
ruta_meta = f'{DATA}/tmdb_meta.parquet'

if all(os.path.exists(r) for r in rutas.values()) and os.path.exists(ruta_meta) \
        and pd.read_parquet(ruta_meta)['tconst'].tolist() == dat['tconst'].tolist():
    E = {m: np.load(r) for m, r in rutas.items()}
    print("Ya existían y coinciden con la tabla actual, se cargan")
else:
    t0 = time.time()
    E = {'clip_txt': enc.texto_clip(dat['sinopsis'].tolist()),
         'distilbert_txt': enc.texto_bert(dat['sinopsis'].tolist())}
    img = []
    for s in range(0, len(dat), 256):
        lote = [rellenar_cuadrado(leer_poster(t)) for t in dat['tconst'].iloc[s:s + 256]]
        img.append(enc.imagen(lote))
    E['clip_img'] = np.concatenate(img)
    for m, r in rutas.items():
        np.save(r, E[m])
    dat.to_parquet(ruta_meta, index=False)
    print(f"Embeddings en {time.time() - t0:.0f} s")

for m, v in E.items():
    print(f"{m:15s} {v.shape}  NaN={np.isnan(v).sum()}  norma media={np.linalg.norm(v, axis=1).mean():.2f}")

## 7. Primera mirada: ¿cuánto cambian los datos?

Tres comparaciones descriptivas, sin modelos todavía (eso es el 08):

1. **Largo de las sinopsis** (tokens de CLIP): las de TMDB suelen ser más cortas.
2. **Misma película, distinta fuente** (control): similitud coseno entre el embedding de MM-IMDB y el de
   TMDB, para imagen y para texto. Cuanto más baja, mayor el efecto de la fuente en esa modalidad.
3. **Alineamiento póster–sinopsis** en cada grupo: recuperación top-k como en el 03. Si cae mucho en las
   recientes, puede ser porque CLIP nunca vio esas películas (se entrenó con datos anteriores a 2021).

In [ ]:
largo = lambda textos: np.array([len(x) for x in enc.tok_clip(list(textos), add_special_tokens=False)['input_ids']])
ctrl_dat = dat[dat['grupo'] == 'control']
largos = {'MM-IMDB (control)': largo(sinopsis_mm.loc[ctrl_dat['fila_hdf5']]),
          'TMDB (control)': largo(ctrl_dat['sinopsis']),
          'TMDB (recientes)': largo(dat.loc[dat['grupo'] == 'reciente', 'sinopsis'])}
for k, v in largos.items():
    print(f"{k:18s} tokens CLIP: mediana {np.median(v):.0f} | p95 {np.percentile(v, 95):.0f}")

# Misma película en las dos fuentes
i_ctrl = np.where(dat['grupo'] == 'control')[0]
filas_ctrl = dat['fila_hdf5'].values[i_ctrl]
print("\nControl, misma película MM-IMDB vs TMDB (similitud coseno):")
for m in ['clip_img', 'clip_txt', 'distilbert_txt']:
    c = cos(E[m][i_ctrl], np.load(f'{EMB}/{m}.npy', mmap_mode='r')[filas_ctrl])
    print(f"  {m:15s} mediana {np.median(c):.3f} | p10 {np.percentile(c, 10):.3f}")

# Alineamiento póster -> sinopsis (CLIP), mismo tamaño de candidatos en cada grupo
def recuperacion(zi, zt, n, rng):
    sub = rng.choice(len(zi), size=n, replace=False)
    a = zi[sub] / np.linalg.norm(zi[sub], axis=1, keepdims=True)
    b = zt[sub] / np.linalg.norm(zt[sub], axis=1, keepdims=True)
    sim = a @ b.T
    rango = (sim > np.diag(sim)[:, None]).sum(1)
    return {f'top-{k}': (rango < k).mean() for k in [1, 5, 10]}

rng = np.random.default_rng(SEED)
n_cand = min(1000, (dat['grupo'] == 'control').sum(), (dat['grupo'] == 'reciente').sum())
print(f"\nRecuperación póster→sinopsis entre {n_cand} candidatos (azar top-1: {1 / n_cand:.2%}):")
zi_mm, zt_mm = (np.load(f'{EMB}/{m}.npy', mmap_mode='r')[filas_ctrl] for m in ['clip_img', 'clip_txt'])
filas_rec = []
for nombre, zi, zt in [('MM-IMDB (control)', zi_mm, zt_mm),
                       ('TMDB (control)', E['clip_img'][i_ctrl], E['clip_txt'][i_ctrl]),
                       ('TMDB (recientes)', E['clip_img'][dat['grupo'] == 'reciente'],
                        E['clip_txt'][dat['grupo'] == 'reciente'])]:
    filas_rec.append({'grupo': nombre, **recuperacion(np.asarray(zi), np.asarray(zt), n_cand, rng)})
pd.DataFrame(filas_rec).set_index('grupo').map('{:.1%}'.format)

## Resumen

Archivos nuevos en Drive:

- `data/tmdb_meta.parquet`: una fila por película incluida (grupo, `tconst`, año, votos, `fila_hdf5` en
  el control, etiquetas `hoy_*`, ids y fechas de TMDB, sinopsis normalizada).
- `embeddings/tmdb_clip_img.npy`, `tmdb_clip_txt.npy`, `tmdb_distilbert_txt.npy`: alineados fila a fila
  con `tmdb_meta.parquet`.
- `data/tmdb/respuestas.jsonl` y `data/tmdb/posters/`: caché de la descarga.

**Siguiente (notebook 08):** cargar `exp2_hoy_artefactos.pt` del 06, estandarizar estos embeddings con
la media y desviación de train (`aplicar_estandarizacion`), predecir con los 15 modelos y comparar test
MM-IMDB vs control vs recientes en los géneros de `generos_comparacion.json`.